In [ ]:
# ============================================
# 04_train_model.ipynb
# Train a model to estimate a player's fair
# percentage of the salary cap.
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import os
os.chdir(r"C:\AI Projects\NFL Player Rating")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# -----------------------------
# Load engineered dataset
# -----------------------------
df = pd.read_csv("data/engineered_dataset.csv")
print("Loaded engineered dataset:", df.shape)

df.head()

In [ ]:
# -----------------------------
# Create custom rating (ZB's formula)
# -----------------------------
df["custom_rating"] = (
    (df["yards_per_game"] * 0.4 +
     df["td_rate"] * 0.3 +
     df["efficiency"] * 0.3)
    * df["pos_value"]
)

# This is the target the model will learn
df["target_cap_percent"] = df["custom_rating"]

df[["player_name", "season", "custom_rating"]].head()

In [ ]:
# -----------------------------
# Select features for the model
# -----------------------------
feature_cols = [
    "pass_yds_pg", "rush_yds_pg", "rec_yds_pg",
    "pass_td_pg", "rush_td_pg", "rec_td_pg",
    "yards_per_attempt", "yards_per_carry", "yards_per_reception",
    "snaps_pg", "touches_pg",
    "pos_value"
]

# Keep only features that exist in the dataset
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]
y = df["target_cap_percent"]

print("Using features:", feature_cols)

In [ ]:
# -----------------------------
# Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# -----------------------------
# Train the model
# -----------------------------
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training complete.")

In [ ]:
# -----------------------------
# Evaluate the model
# -----------------------------
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

In [ ]:
# -----------------------------
# Feature importance
# -----------------------------
importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importances

In [ ]:
# -----------------------------
# Save model predictions
# -----------------------------
df["predicted_cap_percent"] = model.predict(X)

df.to_csv("data/model_output.csv", index=False)
print("Saved model_output.csv with predicted cap percentages.")

df[["player_name", "season", "predicted_cap_percent"]].head()